# 01 Baseline and Optimized Pipeline

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`EXP_CMB_001_KNN_Combined_Baseline.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Combined/scripts/EXP_CMB_001_KNN_Combined_Baseline.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Combined/logs)
- [`EXP_CMB_002_KNN_Combined_Optimized.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Combined/scripts/EXP_CMB_002_KNN_Combined_Optimized.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Combined/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/KNN/Combined/scripts/EXP_CMB_001_KNN_Combined_Baseline.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '23ec197796643ba3e57e01180aadd3e66f98fb2b'}, '05_model_optimize/KNN/Combined/scripts/EXP_CMB_002_KNN_Combined_Optimized.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '999c828788076b5d662c816fdde4611db0e319f7'}}

## EXP_CMB_001_KNN_Combined_Baseline

원본: [05_model_optimize/KNN/Combined/scripts/EXP_CMB_001_KNN_Combined_Baseline.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Combined/scripts/EXP_CMB_001_KNN_Combined_Baseline.py)  
Git blob: `23ec197796643ba3e57e01180aadd3e66f98fb2b`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Combined/logs)

EXP-CMB-001 KNN: Normal versus Combined with 4-writer held-out OOF.

Run from repository root:
 python 05_model_optimize/KNN/Combined/scripts/EXP_CMB_001_KNN_Combined_Baseline.py

Only official training/development validation and writer-separated handwriting are used.
The official Original Test and professor's tests are NEVER read.



In [ ]:
%%archive_source 05_model_optimize/KNN/Combined/scripts/EXP_CMB_001_KNN_Combined_Baseline.py
#!/usr/bin/env python3
"""EXP-CMB-001 KNN: Normal versus Combined with 4-writer held-out OOF.

Run from repository root:
 python 05_model_optimize/KNN/Combined/scripts/EXP_CMB_001_KNN_Combined_Baseline.py

Only official training/development validation and writer-separated handwriting are used.
The official Original Test and professor's tests are NEVER read.
"""
from __future__ import annotations

import csv
import hashlib
import json
import platform
import sys
import time
from pathlib import Path

import numpy as np
import sklearn
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support

ROOT = Path(__file__).resolve().parents[4]
sys.path.insert(0, str(ROOT / "03_experiments" / "model_optimization"))
from common import fixed_split, pipeline, LABELS
from knn import build_baseline

WRITERS = (41, 42, 43, 44)
EXPECTED_HANDMADE_SHA = {
    41: "0a829172a72787c770139c73de34cc8b751181829ff6fe08ec260247b9be5a2f",
    42: "f364fb453d05b4fe2a705c2b8a80683b404608c7c257d664e3ec7ac86932d4c5",
    43: "10db20997c727c1bc1f1299f8e89626fadd3ec9618468d6759c86b0d2ddc4a66",
    44: "0e266a36e7b29edfcca0514836b5c383d13939948b4e335d70f61bb40bdc084d",
}
OUT = ROOT / "05_model_optimize" / "KNN" / "Combined" / "results" / "EXP_CMB_001_baseline_4writer_oof"


def save_csv(path: Path, rows: list[dict], fields: list[str] | None = None) -> None:
    if fields is None:
        fields = list(rows[0]) if rows else []
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def summary(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    ids = np.arange(16)
    p, r, f, n = precision_recall_fscore_support(y_true, y_pred, labels=ids, zero_division=0)
    return {
        "n": int(y_true.size), "correct": int((y_true == y_pred).sum()),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=ids, average="macro", zero_division=0)),
        "per_class": {label: {"precision": float(p[i]), "recall": float(r[i]),
                              "f1": float(f[i]), "support": int(n[i])}
                      for i, label in enumerate(LABELS)},
    }


def save_cm(path: Path, truth: np.ndarray, predictions: np.ndarray) -> None:
    cm = confusion_matrix(truth, predictions, labels=np.arange(16))
    save_csv(path, [{"true\\pred": LABELS[i], **{LABELS[j]: int(cm[i, j]) for j in range(16)}}
                    for i in range(16)])


def load_writer(writer: int) -> tuple[np.ndarray, np.ndarray, str]:
    path = ROOT / "02_data" / "Hand_made" / f"ml26_{writer}.npz"
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != EXPECTED_HANDMADE_SHA[writer]:
        raise ValueError(f"Writer {writer} SHA-256 mismatch: {digest}")
    with np.load(path, allow_pickle=False) as d:
        if set(d.files) != {"img", "label", "label_id"}:
            raise ValueError(f"Unexpected NPZ keys: {path}")
        images = d["img"]
        y = d["label_id"].astype(np.int64)
        text_labels = d["label"]
    if images.shape != (640, 28, 28) or images.dtype != np.uint8 or y.shape != (640,):
        raise ValueError(f"Writer {writer}: unexpected image or label shapes")
    if not np.array_equal(y, np.array([LABELS.index(str(v)) for v in text_labels])):
        raise ValueError(f"Writer {writer}: inconsistent label encoding")
    if not np.all(np.bincount(y, minlength=16) == 40):
        raise ValueError(f"Writer {writer}: class counts differ from 40")
    return images.reshape(640, 784).astype(np.float32), y, digest


def fresh_model():
    knn, scaled = build_baseline()
    if (knn.n_neighbors, knn.weights, knn.p, scaled) != (5, "uniform", 2, True):
        raise ValueError("Baseline definition changed: expected k=5, uniform, Euclidean, pixel/255")
    return pipeline(knn, scaled)


def run():
    OUT.mkdir(parents=True, exist_ok=True)
    x_off_train, y_off_train, x_off_val, y_off_val, data_sha = fixed_split()
    handmade = {w: load_writer(w) for w in WRITERS}
    assert len(x_off_train) == 76800 and len(x_off_val) == 19200
    print(f"Official Train={len(y_off_train)}; Validation={len(y_off_val)}; Hand-made=2560", flush=True)
    print("Official Test is not loaded.", flush=True)

    start = time.perf_counter()
    normal = fresh_model()
    print("[Normal] Fitting official Training only...", flush=True)
    normal.fit(x_off_train, y_off_train)
    normal_fit = time.perf_counter() - start
    normal_off_pred = normal.predict(x_off_val).astype(np.int64)
    normal_official = summary(y_off_val, normal_off_pred)
    print(f"[Normal] Official Validation={normal_official['accuracy']:.6f}; "
          f"MacroF1={normal_official['macro_f1']:.6f}", flush=True)
    if normal_official["correct"] != 18488:
        raise RuntimeError("Normal official baseline did not reproduce EXP-001 (18488/19200). "
                           "Check environment/data and stop before Combined interpretation.")

    normal_preds = {w: normal.predict(handmade[w][0]).astype(np.int64) for w in WRITERS}
    combined_preds = {}
    fold_rows = []
    official_rows = [{"model": "normal", "heldout_writer": "all", **{k: normal_official[k] for k in ("n", "correct", "accuracy", "macro_f1")}}]

    for heldout in WRITERS:
        fit_writers = [w for w in WRITERS if w != heldout]
        x_extra = np.concatenate([handmade[w][0] for w in fit_writers], axis=0)
        y_extra = np.concatenate([handmade[w][1] for w in fit_writers], axis=0)
        x_train = np.concatenate([x_off_train, x_extra], axis=0)
        y_train = np.concatenate([y_off_train, y_extra], axis=0)
        if len(x_train) != 78720 or len(y_extra) != 1920:
            raise RuntimeError("Combined training size mismatch")
        if heldout in fit_writers:
            raise RuntimeError("Hand-made writer leakage")
        print(f"\n[Fold heldout={heldout}] Train writers={fit_writers}; rows={len(y_train)}", flush=True)
        model = fresh_model()
        t = time.perf_counter()
        model.fit(x_train, y_train)
        fit_s = time.perf_counter() - t
        t = time.perf_counter()
        pred = model.predict(handmade[heldout][0]).astype(np.int64)
        handwritten_s = time.perf_counter() - t
        t = time.perf_counter()
        off_pred = model.predict(x_off_val).astype(np.int64)
        official_s = time.perf_counter() - t
        combined_preds[heldout] = pred
        hand_result = summary(handmade[heldout][1], pred)
        normal_result = summary(handmade[heldout][1], normal_preds[heldout])
        off_result = summary(y_off_val, off_pred)
        fold_rows.append({
            "heldout_writer": heldout, "train_writers": ",".join(map(str, fit_writers)),
            "official_train_n": 76800, "handmade_train_n": 1920, "train_n": 78720,
            "heldout_n": 640, "normal_accuracy": normal_result["accuracy"],
            "normal_macro_f1": normal_result["macro_f1"],
            "combined_accuracy": hand_result["accuracy"],
            "combined_macro_f1": hand_result["macro_f1"],
            "combined_correct": hand_result["correct"],
            "combined_official_val_accuracy": off_result["accuracy"],
            "combined_official_val_macro_f1": off_result["macro_f1"],
            "fit_s": round(fit_s, 3), "handmade_predict_s": round(handwritten_s, 3),
            "official_val_predict_s": round(official_s, 3),
        })
        official_rows.append({"model": "combined", "heldout_writer": heldout,
                              **{k: off_result[k] for k in ("n", "correct", "accuracy", "macro_f1")}})
        print(f"   Hand-made Normal={normal_result['accuracy']:.6f} / "
              f"Combined={hand_result['accuracy']:.6f}, "
              f"Official Validation={off_result['accuracy']:.6f}", flush=True)
        save_csv(OUT / "fold_results_partial.csv", fold_rows)
        save_csv(OUT / "official_validation_partial.csv", official_rows)

    truth = np.concatenate([handmade[w][1] for w in WRITERS])
    normal_oof = np.concatenate([normal_preds[w] for w in WRITERS])
    combined_oof = np.concatenate([combined_preds[w] for w in WRITERS])
    nres, cres = summary(truth, normal_oof), summary(truth, combined_oof)
    paired = {
        "corrected": int(((normal_oof != truth) & (combined_oof == truth)).sum()),
        "new_errors": int(((normal_oof == truth) & (combined_oof != truth)).sum()),
        "unchanged_correct": int(((normal_oof == truth) & (combined_oof == truth)).sum()),
        "unchanged_wrong": int(((normal_oof != truth) & (combined_oof != truth)).sum()),
    }
    predictions = []
    for offset, w in enumerate(WRITERS):
        y = handmade[w][1]
        for i in range(640):
            idx = offset * 640 + i
            predictions.append({"writer": w, "index": i, "true": LABELS[int(y[i])],
                                "normal_pred": LABELS[int(normal_oof[idx])],
                                "combined_pred": LABELS[int(combined_oof[idx])],
                                "normal_correct": int(normal_oof[idx] == y[i]),
                                "combined_correct": int(combined_oof[idx] == y[i])})

    save_csv(OUT / "fold_results.csv", fold_rows)
    save_csv(OUT / "official_validation.csv", official_rows)
    save_csv(OUT / "oof_predictions.csv", predictions)
    save_cm(OUT / "normal_oof_confusion_matrix.csv", truth, normal_oof)
    save_cm(OUT / "combined_oof_confusion_matrix.csv", truth, combined_oof)
    save_csv(OUT / "per_class_oof.csv", [
        {"class": label, "normal_recall": nres["per_class"][label]["recall"],
         "normal_f1": nres["per_class"][label]["f1"],
         "combined_recall": cres["per_class"][label]["recall"],
         "combined_f1": cres["per_class"][label]["f1"],
         "support": nres["per_class"][label]["support"]}
        for label in LABELS
    ])
    report = {
        "experiment": "EXP-CMB-001 KNN Baseline / 4-Writer OOF",
        "official_train": 76800, "official_validation": 19200,
        "combined_train_per_fold": 78720, "handmade_train_per_fold": 1920,
        "heldout_per_fold": 640, "handmade_total": 2560,
        "data_sha256": data_sha,
        "handmade_sha256": {str(w): handmade[w][2] for w in WRITERS},
        "model": "KNeighborsClassifier(n_neighbors=5, weights='uniform', p=2)",
        "scaling": "pixel / 255",
        "official_test_used": False,
        "normal_oof": nres, "combined_oof": cres,
        "paired_handmade": paired,
        "folds": fold_rows, "official_validation_by_model": official_rows,
        "normal_fit_seconds": normal_fit,
        "environment": {"python": platform.python_version(), "numpy": np.__version__,
                        "sklearn": sklearn.__version__},
        "note": "Writer-held-out development OOF; no official Test and no heldout-writer training leakage",
    }
    (OUT / "metrics.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
    print("\n4-WRITER OOF SUMMARY", flush=True)
    print(f"Normal   Accuracy={nres['accuracy']:.6f}, Macro F1={nres['macro_f1']:.6f} "
          f"({nres['correct']}/2560)", flush=True)
    print(f"Combined Accuracy={cres['accuracy']:.6f}, Macro F1={cres['macro_f1']:.6f} "
          f"({cres['correct']}/2560)", flush=True)
    print(f"Change={100*(cres['accuracy'] - nres['accuracy']):+.4f} percentage points", flush=True)
    print(f"Paired={paired}", flush=True)
    print(f"Saved: {OUT}", flush=True)


if __name__ == "__main__":
    run()


## EXP_CMB_002_KNN_Combined_Optimized

원본: [05_model_optimize/KNN/Combined/scripts/EXP_CMB_002_KNN_Combined_Optimized.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Combined/scripts/EXP_CMB_002_KNN_Combined_Optimized.py)  
Git blob: `999c828788076b5d662c816fdde4611db0e319f7`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/KNN/Combined/logs)

EXP-CMB-002: frozen optimized KNN Normal versus +3-writer Combined, 4-writer OOF.

Run from repository root:
 python 05_model_optimize/KNN/Combined/scripts/EXP_CMB_002_KNN_Combined_Optimized.py

Official Test is NEVER loaded. Every held-out writer is excluded from that fold's
training rows, but the frozen Normal configuration (including weak classes) was
selected with the same 4-writer development set in earlier EXP-013/014. Thus the
OOF result is *development OOF*, NOT an independent unbiased g



In [ ]:
%%archive_source 05_model_optimize/KNN/Combined/scripts/EXP_CMB_002_KNN_Combined_Optimized.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""EXP-CMB-002: frozen optimized KNN Normal versus +3-writer Combined, 4-writer OOF.

Run from repository root:
 python 05_model_optimize/KNN/Combined/scripts/EXP_CMB_002_KNN_Combined_Optimized.py

Official Test is NEVER loaded. Every held-out writer is excluded from that fold's
training rows, but the frozen Normal configuration (including weak classes) was
selected with the same 4-writer development set in earlier EXP-013/014. Thus the
OOF result is *development OOF*, NOT an independent unbiased generalization test.
"""
from __future__ import annotations

import json
import platform
import sys
import time
from pathlib import Path

import numpy as np
import sklearn
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix

ROOT = Path(__file__).resolve().parents[4]
NORMAL = ROOT / "05_model_optimize" / "KNN" / "Normal" / "scripts"
COMBINED = ROOT / "05_model_optimize" / "KNN" / "Combined" / "scripts"
sys.path.insert(0, str(NORMAL))
sys.path.insert(0, str(COMBINED))
sys.path.insert(0, str(ROOT / "03_experiments" / "model_optimization"))

from common import LABELS, fixed_split
from EXP004_KNN_Normal_CoM import BatchedCoM
from final_knn_normal import assemble_training, prepare_features, EXPECTED_VALIDATION_CORRECT
from EXP_CMB_001_KNN_Combined_Baseline import (
    WRITERS, load_writer, summary, save_csv, save_cm,
)

OUT = ROOT / "05_model_optimize" / "KNN" / "Combined" / "results" / "EXP_CMB_002_optimized_4writer_oof"
EXPECTED_NORMAL_HANDMADE = 2381  # EXP-014 targeted_10 / EXP-015 k3_distance


def fresh_knn():
    return KNeighborsClassifier(n_neighbors=3, weights="distance", metric="euclidean",
                                algorithm="brute", n_jobs=-1)


def paired_counts(y, before, after):
    return {
        "corrected": int(((before != y) & (after == y)).sum()),
        "new_errors": int(((before == y) & (after != y)).sum()),
        "unchanged_correct": int(((before == y) & (after == y)).sum()),
        "unchanged_wrong": int(((before != y) & (after != y)).sum()),
    }


def run():
    OUT.mkdir(parents=True, exist_ok=True)
    xtr, ytr, xva, yva, sha = fixed_split()
    if (len(ytr), len(yva)) != (76800, 19200):
        raise RuntimeError("Unexpected official development split")
    print("Official Training=76800; Official Validation=19200; Official Test NOT loaded", flush=True)
    handmade = {w: load_writer(w) for w in WRITERS}
    com = BatchedCoM()
    t0 = time.perf_counter()

    x_train_opt, y_train_opt = assemble_training(xtr, ytr, com)
    del xtr, ytr
    if x_train_opt.shape != (107520, 1108):
        raise RuntimeError("Frozen Normal training feature count mismatch")
    print("[Preprocessing] Official Validation and four writer sets...", flush=True)
    fv = prepare_features(xva, com, "official Validation")
    del xva
    fw = {w: prepare_features(handmade[w][0], com, f"handmade writer {w}") for w in WRITERS}

    print("[Optimized Normal] Official Training + official augmentation only...", flush=True)
    normal = fresh_knn()
    normal.fit(x_train_opt, y_train_opt)
    normal_off_pred = normal.predict(fv).astype(np.int64)
    normal_off = summary(yva, normal_off_pred)
    if normal_off["correct"] != EXPECTED_VALIDATION_CORRECT:
        raise RuntimeError(f"Frozen Normal Validation mismatch: {normal_off['correct']} vs 18851")
    normal_pred = {w: normal.predict(fw[w]).astype(np.int64) for w in WRITERS}
    truth = np.concatenate([handmade[w][1] for w in WRITERS])
    n_oof = np.concatenate([normal_pred[w] for w in WRITERS])
    n_result = summary(truth, n_oof)
    if n_result["correct"] != EXPECTED_NORMAL_HANDMADE:
        raise RuntimeError(f"Frozen Normal handwriting mismatch: {n_result['correct']} vs 2381")
    print(f"Normal EXACTLY REPRODUCED: Official={normal_off['accuracy']:.6f}; "
          f"Hand-made={n_result['accuracy']:.6f}", flush=True)
    del normal

    combined_pred = {}
    fold_rows = []
    official_rows = [{"variant": "optimized_normal", "heldout_writer": "all",
                      "n": normal_off["n"], "correct": normal_off["correct"],
                      "accuracy": normal_off["accuracy"], "macro_f1": normal_off["macro_f1"]}]

    for heldout in WRITERS:
        other = [w for w in WRITERS if w != heldout]
        extra_x = np.concatenate([fw[w] for w in other], axis=0)
        extra_y = np.concatenate([handmade[w][1] for w in other], axis=0)
        if extra_x.shape != (1920, 1108) or len(extra_y) != 1920:
            raise RuntimeError("Wrong 3-writer training rows")
        fit_x = np.concatenate([x_train_opt, extra_x], axis=0)
        fit_y = np.concatenate([y_train_opt, extra_y], axis=0)
        if fit_x.shape != (109440, 1108) or heldout in other:
            raise RuntimeError("Fit shape or held-out writer leakage")
        del extra_x, extra_y
        print(f"\n[Fold heldout={heldout}] Train writers={other}; "
              f"107520 official-derived + 1920 writer rows = 109440", flush=True)
        model = fresh_knn()
        t = time.perf_counter()
        model.fit(fit_x, fit_y)
        fit_seconds = time.perf_counter() - t
        del fit_x, fit_y
        t = time.perf_counter()
        pred_hm = model.predict(fw[heldout]).astype(np.int64)
        hand_predict_seconds = time.perf_counter() - t
        t = time.perf_counter()
        pred_off = model.predict(fv).astype(np.int64)
        off_predict_seconds = time.perf_counter() - t
        del model
        combined_pred[heldout] = pred_hm
        cr = summary(handmade[heldout][1], pred_hm)
        nr = summary(handmade[heldout][1], normal_pred[heldout])
        ov = summary(yva, pred_off)
        fold_rows.append({
            "heldout_writer": heldout, "train_writers": ",".join(map(str, other)),
            "official_derived_train": 107520, "handmade_train": 1920,
            "combined_train_n": 109440, "heldout_n": 640,
            "normal_correct": nr["correct"], "normal_accuracy": nr["accuracy"],
            "normal_macro_f1": nr["macro_f1"], "combined_correct": cr["correct"],
            "combined_accuracy": cr["accuracy"], "combined_macro_f1": cr["macro_f1"],
            "combined_official_accuracy": ov["accuracy"],
            "combined_official_macro_f1": ov["macro_f1"],
            "fit_s": round(fit_seconds, 3),
            "handmade_predict_s": round(hand_predict_seconds, 3),
            "official_predict_s": round(off_predict_seconds, 3),
        })
        official_rows.append({"variant": "combined", "heldout_writer": heldout,
                              "n": ov["n"], "correct": ov["correct"],
                              "accuracy": ov["accuracy"], "macro_f1": ov["macro_f1"]})
        save_csv(OUT / "fold_results_partial.csv", fold_rows)
        save_csv(OUT / "official_validation_partial.csv", official_rows)
        print(f"   Hand-made Normal={nr['accuracy']:.6f} / Combined={cr['accuracy']:.6f}; "
              f"Official Validation={ov['accuracy']:.6f}", flush=True)

    c_oof = np.concatenate([combined_pred[w] for w in WRITERS])
    c_result = summary(truth, c_oof)
    paired = paired_counts(truth, n_oof, c_oof)
    preds = []
    for offset, w in enumerate(WRITERS):
        for i in range(640):
            j = offset * 640 + i
            preds.append({"writer": w, "index": i, "true": LABELS[int(truth[j])],
                          "normal_pred": LABELS[int(n_oof[j])],
                          "combined_pred": LABELS[int(c_oof[j])],
                          "normal_correct": int(n_oof[j] == truth[j]),
                          "combined_correct": int(c_oof[j] == truth[j])})
    save_csv(OUT / "fold_results.csv", fold_rows)
    save_csv(OUT / "official_validation.csv", official_rows)
    save_csv(OUT / "oof_predictions.csv", preds)
    save_cm(OUT / "normal_oof_confusion_matrix.csv", truth, n_oof)
    save_cm(OUT / "combined_oof_confusion_matrix.csv", truth, c_oof)
    save_csv(OUT / "per_class_oof.csv", [
        {"class": label, "support": n_result["per_class"][label]["support"],
         "normal_recall": n_result["per_class"][label]["recall"],
         "normal_f1": n_result["per_class"][label]["f1"],
         "combined_recall": c_result["per_class"][label]["recall"],
         "combined_f1": c_result["per_class"][label]["f1"]}
        for label in LABELS])
    report = {
        "experiment": "EXP-CMB-002: optimized Normal vs raw 3-writer Combined, development 4-writer OOF",
        "official_dataset_sha256": sha,
        "handmade_sha256": {str(w): handmade[w][2] for w in WRITERS},
        "official_train_source": 76800, "official_derived_train": 107520,
        "handmade_train_per_fold": 1920, "combined_train_per_fold": 109440,
        "official_val": 19200, "heldout_per_fold": 640,
        "frozen_model": {"k": 3, "weights": "distance", "metric": "euclidean",
                         "preprocessing": "CoM -> Deskew", "features": "Pixel 20% + HOG 80%",
                         "base_aug": "30% size+stroke, strength 0.44",
                         "additional_aug": "D/B/5/3 7680 official-training-derived rows"},
        "normal_oof": n_result, "combined_oof": c_result,
        "normal_official_validation": normal_off,
        "official_validation_per_fold": official_rows,
        "folds": fold_rows, "paired_handmade": paired,
        "official_test_used": False,
        "warning": "All 4 Hand-made writers were used to choose the frozen Normal feature and targeted classes in earlier EXP-013/014. Despite per-fold writer exclusion from FIT, this is development OOF and cannot be claimed independent/unbiased writer generalization.",
        "environment": {"python": platform.python_version(), "numpy": np.__version__,
                        "sklearn": sklearn.__version__},
        "elapsed_seconds": round(time.perf_counter()-t0, 3),
    }
    (OUT / "metrics.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print("\n4-WRITER DEVELOPMENT OOF SUMMARY", flush=True)
    print(f"Optimized Normal   Accuracy={n_result['accuracy']:.6f}, "
          f"MacroF1={n_result['macro_f1']:.6f} ({n_result['correct']}/2560)", flush=True)
    print(f"Optimized Combined Accuracy={c_result['accuracy']:.6f}, "
          f"MacroF1={c_result['macro_f1']:.6f} ({c_result['correct']}/2560)", flush=True)
    print(f"Change={100*(c_result['accuracy']-n_result['accuracy']):+.4f} percentage points", flush=True)
    print(f"Paired={paired}", flush=True)
    print(f"Saved: {OUT}", flush=True)
    print("No official Test was loaded. OOF is development-only due to prior Hand-made model selection.", flush=True)


if __name__ == "__main__":
    run()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")